# RabTech Academy - Task 03
## Data Ingestion, Cleaning & Preprocessing with Pandas

This notebook cleans the included messy business dataset and exports `clean_dataset.csv`.


In [ ]:
import pandas as pd
import numpy as np
from IPython.display import display


## 1. Load the raw Excel dataset


In [ ]:
df = pd.read_excel("raw_dataset.xlsx")
print("Dataset loaded successfully.")
print("Shape:", df.shape)
display(df.head())


## 2. Inspect the raw dataset


In [ ]:
df.info()
print("\nMissing values:")
display(df.isnull().sum().to_frame("Missing Values"))
print("\nDuplicate rows:", df.duplicated().sum())


## 3. Standardize data types and category values


In [ ]:
df["Category"] = df["Category"].astype("string").str.strip().str.title()
df["Order_Date"] = pd.to_datetime(df["Order_Date"], errors="coerce")
for col in ["Sales","Profit","Quantity","Customer_Age","Discount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
print(df.dtypes)


## 4. Remove duplicate records


In [ ]:
before = len(df)
df = df.drop_duplicates().copy()
print("Duplicates removed:", before-len(df))


## 5. Handle missing values


In [ ]:
for col in ["Sales","Profit","Quantity","Customer_Age","Discount"]:
    df[col] = df[col].fillna(df[col].median())
median_date = df["Order_Date"].dropna().sort_values().iloc[len(df["Order_Date"].dropna())//2]
df["Order_Date"] = df["Order_Date"].fillna(median_date)
df["Category"] = df["Category"].fillna(df["Category"].mode()[0])
print("Remaining missing values:")
display(df.isnull().sum().to_frame("Missing Values"))


## 6. Feature engineering


In [ ]:
df["Year"] = df["Order_Date"].dt.year
df["Month"] = df["Order_Date"].dt.month
df["Month_Name"] = df["Order_Date"].dt.month_name()
df["Profit_Margin"] = np.where(df["Sales"] != 0, (df["Profit"]/df["Sales"])*100, 0)
display(df.head())


## 7. Handle Sales outliers using IQR


In [ ]:
q1 = df["Sales"].quantile(.25)
q3 = df["Sales"].quantile(.75)
iqr = q3-q1
lower = q1-1.5*iqr
upper = q3+1.5*iqr
before = len(df)
df = df[(df["Sales"]>=lower)&(df["Sales"]<=upper)].copy()
print("IQR limits:", round(lower,2), "to", round(upper,2))
print("Outlier rows removed:", before-len(df))


## 8. Verify the cleaned dataset


In [ ]:
print("Final shape:", df.shape)
print("Remaining duplicates:", df.duplicated().sum())
print("\nRemaining missing values:")
display(df.isnull().sum().to_frame("Missing Values"))
print("\nFinal data types:")
print(df.dtypes)


## 9. Export clean dataset


In [ ]:
df.to_csv("clean_dataset.csv", index=False)
print("Saved successfully as clean_dataset.csv")


## Conclusion
- Loaded a 10,000+ row Excel business dataset.
- Handled missing values and duplicate records.
- Corrected data types and inconsistent category formatting.
- Created Year, Month, Month_Name and Profit_Margin features.
- Handled Sales outliers using the IQR method.
- Exported the cleaned dataset as `clean_dataset.csv`.
